# VSHORAD Strategic Training Pipeline

**Tier**: Strategic (A100 40/80GB)

| Component | Model | Resolution | Epochs |
|-----------|-------|------------|--------|
| Detection | YOLOv8l | 1280px | 120 |
| Classification | Swin-Base | 384px | 100 |

Training pipeline for the highest-accuracy configuration.
Requires A100 GPU (Colab Pro).

## 1 Setup & Environment

In [ ]:
# 1.1 Installation

!pip uninstall -y numpy
!pip install numpy==1.26.4 -q
!pip install ultralytics==8.3.0 timm==1.0.11 -q
!pip install codecarbon==2.4.2 albumentations==1.4.18 lapx -q

print(" Dependencies installed.")

In [ ]:
# 1.2 Imports

import os, sys, json, yaml, time, random, shutil, zipfile, datetime, warnings
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms, datasets

import timm
from timm.data import Mixup
import albumentations as A
from albumentations.pytorch import ToTensorV2

from ultralytics import YOLO
from codecarbon import EmissionsTracker

from sklearn.metrics import (
  confusion_matrix, classification_report,
  precision_recall_fscore_support, accuracy_score
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print(" All imports loaded!")

In [ ]:
# 1.3 Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

print(" Google Drive mounted.")

In [ ]:
# 1.4 GPU verification

import torch
if torch.cuda.is_available():
  gpu = torch.cuda.get_device_name(0)
  vram = torch.cuda.get_device_properties(0).total_mem / 1e9
  print(f" GPU: {gpu} ({vram:.0f} GB)")
  DEVICE = 'cuda'
else:
  print(" No GPU — training will be very slow.")
  DEVICE = 'cpu'

## 2 Configuration

In [ ]:
# 2.1 Paths

DRIVE_ROOT = Path('/content/drive/MyDrive/Inżynierka')
DATASETS_DIR = DRIVE_ROOT / 'Datasety Finalne' / 'Datasety_Zbalansowane'

ZIP_YOLO = DATASETS_DIR / 'Skydetect_Balanced_Yolo_Dataset.zip'
ZIP_SWIN = DATASETS_DIR / 'Skydetect_Balanced_Swin_Dataset.zip'

OUTPUT_ROOT = DRIVE_ROOT / 'Strategic_components'
CHECKPOINTS_DIR = OUTPUT_ROOT / 'checkpoints'
YOLO_CHECKPOINT_DIR = CHECKPOINTS_DIR / 'yolo' / 'strategic_yolov8l_1280'
SWIN_CHECKPOINT_DIR = CHECKPOINTS_DIR / 'swin' / 'strategic_swin_base_384'
LOGS_DIR = OUTPUT_ROOT / 'logs'
REPORTS_DIR = OUTPUT_ROOT / 'reports'
XAI_DIR = OUTPUT_ROOT / 'xai_visualizations'
EXPORTS_DIR = OUTPUT_ROOT / 'exports'
SYSTEM_DIR = OUTPUT_ROOT / 'integrated_system'
VIDEO_RESULTS_DIR = OUTPUT_ROOT / 'video_results'

LOCAL_ROOT = Path('/content/datasets')
LOCAL_YOLO = LOCAL_ROOT / 'yolo'
LOCAL_SWIN = LOCAL_ROOT / 'swin'

for folder in [OUTPUT_ROOT, CHECKPOINTS_DIR, YOLO_CHECKPOINT_DIR, SWIN_CHECKPOINT_DIR,
        LOGS_DIR, REPORTS_DIR, XAI_DIR, EXPORTS_DIR, SYSTEM_DIR, VIDEO_RESULTS_DIR,
        LOCAL_ROOT, LOCAL_YOLO, LOCAL_SWIN]:
  folder.mkdir(parents=True, exist_ok=True)

print(" Folder structure:")
print(f"  Datasets ZIP: {DATASETS_DIR}")
print(f"  Output: {OUTPUT_ROOT}")
print(f"  Local cache: {LOCAL_ROOT}")

In [ ]:
# 2.2 YOLO hyperparameters

YOLO_CONFIG = {
  'model': 'yolov8l.pt', 'task': 'detect',
  'epochs': 120, 'patience': 25, 'batch': 32, 'imgsz': 1280, 'device': 0,
  'optimizer': 'AdamW', 'lr0': 0.002, 'lrf': 0.01, 'momentum': 0.937,
  'weight_decay': 0.0005, 'warmup_epochs': 5, 'warmup_momentum': 0.8, 'warmup_bias_lr': 0.1,
  'box': 7.5, 'cls': 0.5, 'dfl': 1.5,
  'mosaic': 1.0, 'mixup': 0.15, 'copy_paste': 0.1, 'degrees': 5.0, 'translate': 0.1,
  'scale': 0.5, 'shear': 2.0, 'perspective': 0.0, 'flipud': 0.0, 'fliplr': 0.5,
  'hsv_h': 0.015, 'hsv_s': 0.5, 'hsv_v': 0.4, 'erasing': 0.2,
  'workers': 8, 'project': str(CHECKPOINTS_DIR / 'yolo'),
  'name': 'strategic_yolov8l_1280', 'exist_ok': True, 'pretrained': True,
  'verbose': True, 'seed': 42, 'save_period': 10,
}

YOLO_CLASSES = {
  0: 'Fighter', 1: 'Helicopter', 2: 'Transport', 3: 'Bomber',
  4: 'Special', 5: 'UAV_Fixed', 6: 'UAV_Rotor', 7: 'Missile',
  8: 'Civilian', 9: 'Birds', 10: 'Decoys', 11: 'Unidentified'
}

print(f" YOLO: {YOLO_CONFIG['model']}, {YOLO_CONFIG['epochs']} epochs, {YOLO_CONFIG['imgsz']}px")

In [ ]:
# 2.3 Swin hyperparameters

SWIN_CONFIG = {
  'model_name': 'swin_base_patch4_window12_384', 'img_size': 384,
  'num_classes': 56, 'pretrained': True,
  'epochs': 100, 'batch_size': 64, 'num_workers': 8,
  'optimizer': 'AdamW', 'lr': 2e-4, 'weight_decay': 0.05,
  'warmup_epochs': 5, 'min_lr': 1e-6,
  'drop_path_rate': 0.2, 'label_smoothing': 0.1,
  'use_ema': True, 'ema_decay': 0.9999,
  'mixup_alpha': 0.8, 'cutmix_alpha': 1.0, 'mixup_prob': 0.5,
  'save_period': 5, 'seed': 42,
}

print(f" Swin: {SWIN_CONFIG['model_name']}, {SWIN_CONFIG['epochs']} epochs, {SWIN_CONFIG['img_size']}px")

In [ ]:
# 2.3.1 Custom ByteTrack config (improved track retention)

import yaml

BYTETRACK_CUSTOM_CONFIG = {
  'tracker_type': 'bytetrack',
  'track_high_thresh': 0.45,  # Lowered from 0.5 for better track retention
  'track_low_thresh': 0.1,
  'new_track_thresh': 0.55,   # Lowered from 0.6 for faster new track creation
  'track_buffer': 90,      # INCREASZONY z 30 do 90 (~3s @ 30fps)
  'match_thresh': 0.92,     # Increased from 0.8 for better matching after occlusion
  'fuse_score': True,
}

# Save to file
bytetrack_custom_path = '/content/bytetrack_custom.yaml'
with open(bytetrack_custom_path, 'w') as f:
  yaml.dump(BYTETRACK_CUSTOM_CONFIG, f)

print(f" Custom ByteTrack config saved: {bytetrack_custom_path}")
print(f"  track_buffer: {BYTETRACK_CUSTOM_CONFIG['track_buffer']} frames")
print(f"  match_thresh: {BYTETRACK_CUSTOM_CONFIG['match_thresh']}")

In [ ]:
# 2.4 Feedback Loop config (v6 - with unidentified detection)

SWIN_TO_YOLO_CATEGORY = {
  # FIGHTER (YOLO 0)
  'F-16': 0, 'F-15': 0, 'F-22': 0, 'F-35': 0, 'F-14': 0, 'F-4': 0, 'FA-18': 0,
  'EUROFIGHTER': 0, 'GRIPEN': 0, 'RAFALE': 0, 'MIRAGE-2000': 0, 'J-10': 0,
  'J-20': 0, 'J-31': 0, 'JF-17': 0, 'MIG-31': 0, 'Fulcrum_family': 0,
  'Flanker_family': 0, 'SU-57': 0, 'YF-23': 0, 'F-117': 0,
  # HELICOPTER (YOLO 1)
  'AH-64': 1, 'AH-1': 1, 'MI-24': 1, 'MI-28': 1, 'KA-52': 1, 'MI-8': 1,
  'CH-47': 1, 'UH60M': 1, 'TIGER': 1, 'Z10': 1,
  # TRANSPORT (YOLO 2)
  'C130': 2, 'C17': 2, 'C5': 2, 'AN-124': 2, 'AN-26': 2, 'IL-76': 2, 'Y20': 2,
  # BOMBER (YOLO 3)
  'B1': 3, 'B2': 3, 'B52': 3, 'TU-160': 3, 'TU-95': 3, 'VULCAN': 3, 'H6': 3, 'SU-34': 3,
  # SPECIAL (YOLO 4)
  'E2': 4, 'E3': 4, 'E7': 4, 'KC-135': 4, 'U2': 4, 'SR-71': 4, 'V22': 4,
  'A-10': 4, 'AV-8B': 4, 'SU-25': 4,
}

SWIN_SUPPORTED_YOLO_CLASSES = {0, 1, 2, 3, 4}
SWIN_UNKNOWN_YOLO_CLASSES = {5, 6, 7, 8, 9, 10, 11}

FEEDBACK_CONFIG = {
  # Swin classification
  'always_classify': [0, 3],      # Fighter, Bomber
  'classify_if_uncertain': [1, 2, 4], # Helicopter, Transport, Special
  'never_classify': [5, 6, 7, 8, 9, 10, 11],
  'uncertainty_threshold': 0.70,

  # SWIN EXPONENTIAL MOVING AVERAGE (EMA)
  'swin_ema_alpha': 0.4,
  'swin_ema_min_samples': 2,

  # Reclassification
  'reclassify_interval': 25,

  # BBOX SMOOTHING
  'bbox_smoothing': True,
  'bbox_alpha': 0.6,

  # RE-ID
  'reid_enabled': True,
  'reid_max_frames': 45,
  'reid_max_distance': 200,
  'reid_velocity_mult': 1.5,
  'reid_same_class_required': True,

  # TRACK PERSISTENCE
  'track_hold_frames': 60,
  'min_bbox_size': 20,

  # FUSION
  'swin_override_threshold': 0.80,
  'yolo_weak_threshold': 0.50,

  # STABILIZACJA YOLO
  'yolo_stable_frames': 8,

  # UNIDENTIFIED DETECTION (NOWE!)
  'unidentified_window': 30,    # Time window (frames) for analyzing changes
  'unidentified_max_changes': 10, # >10 zmian w oknie = Unidentified
}

print(f" Feedback config v6 (UNIDENTIFIED DETECTION):")
print(f"  Swin EMA alpha: {FEEDBACK_CONFIG['swin_ema_alpha']}")
print(f"  Re-ID max frames: {FEEDBACK_CONFIG['reid_max_frames']}")
print(f"  Unidentified: >{FEEDBACK_CONFIG['unidentified_max_changes']} zmian w {FEEDBACK_CONFIG['unidentified_window']} klatkach")

## 3 Dataset Preparation

In [ ]:
# 3.1 Helper functions

def unpack_dataset_with_progress(zip_path: Path, target_dir: Path) -> bool:
  if not zip_path.exists():
    print(f" ERROR: Not found: {zip_path}")
    return False
  if target_dir.exists() and any(target_dir.iterdir()):
    print(f" {target_dir.name} already exists. Skipping.")
    return True
  print(f" Extracting: {zip_path.name}...")
  target_dir.mkdir(parents=True, exist_ok=True)
  try:
    with zipfile.ZipFile(zip_path, 'r') as zf:
      file_list = zf.infolist()
      total_size = sum(f.file_size for f in file_list)
      with tqdm(total=total_size, unit='B', unit_scale=True, desc=f"Extracting") as pbar:
        for member in file_list:
          zf.extract(member, target_dir)
          pbar.update(member.file_size)
    print(f" Extracted do: {target_dir}")
    return True
  except Exception as e:
    print(f" Error: {e}")
    return False

def count_yolo_samples(root_dir: Path) -> Dict[str, int]:
  counts = {}
  for split in ['train', 'val', 'test']:
    patterns = [root_dir / 'images' / split, *list(root_dir.rglob(f'images/{split}'))]
    for img_dir in patterns:
      if img_dir.exists() and img_dir.is_dir():
        counts[split] = len(list(img_dir.glob('*.jpg'))) + len(list(img_dir.glob('*.png')))
        break
    else:
      counts[split] = 0
  return counts

def count_swin_samples(root_dir: Path) -> Dict[str, Dict[str, int]]:
  result = {}
  for split in ['train', 'val', 'test']:
    patterns = [root_dir / split, *list(root_dir.rglob(split))]
    for d in patterns:
      if d.exists() and d.is_dir() and any(d.iterdir()):
        result[split] = {}
        for cls_dir in sorted(d.iterdir()):
          if cls_dir.is_dir():
            count = len(list(cls_dir.glob('*.jpg'))) + len(list(cls_dir.glob('*.png')))
            result[split][cls_dir.name] = count
        break
  return result

print(" Funkcje pomocnicze zdefiniowane.")

In [ ]:
# 3.2 Extract datasets

print(" Checking datasets...")
print(f"  YOLO ZIP: {ZIP_YOLO}")
print(f"  SWIN ZIP: {ZIP_SWIN}")
print()

yolo_ok = unpack_dataset_with_progress(ZIP_YOLO, LOCAL_YOLO)
swin_ok = unpack_dataset_with_progress(ZIP_SWIN, LOCAL_SWIN)

if yolo_ok and swin_ok:
  print("\n Both datasets ready.")
else:
  print("\n Some datasets were not extracted.")

In [ ]:
# 3.3 Resolve actual dataset paths

def find_yolo_data_yaml(root: Path):
  candidates = list(root.rglob('data.yaml')) + list(root.rglob('*.yaml'))
  for c in candidates:
    if 'data' in c.name.lower():
      return c
  return candidates[0] if candidates else None

def find_swin_train_dir(root: Path):
  for c in list(root.rglob('train')):
    if c.is_dir() and any(c.iterdir()):
      subdirs = [d for d in c.iterdir() if d.is_dir()]
      if subdirs:
        return c
  return None

YOLO_DATA_YAML = find_yolo_data_yaml(LOCAL_YOLO)
SWIN_TRAIN_DIR = find_swin_train_dir(LOCAL_SWIN)
SWIN_VAL_DIR = SWIN_TRAIN_DIR.parent / 'val' if SWIN_TRAIN_DIR else None
SWIN_TEST_DIR = SWIN_TRAIN_DIR.parent / 'test' if SWIN_TRAIN_DIR else None

print(" Found paths:")
print(f"  YOLO data.yaml: {YOLO_DATA_YAML}")
print(f"  Swin train: {SWIN_TRAIN_DIR}")
print(f"  Swin val: {SWIN_VAL_DIR}")

In [ ]:
# Create data.yaml for YOLO
YOLO_DATASET_ROOT = Path('/content/datasets/yolo/Skydetect_Balanced_Yolo_Dataset')

yolo_yaml_content = {
  'path': str(YOLO_DATASET_ROOT),
  'train': 'images/train',
  'val': 'images/val',
  'test': 'images/test',
  'names': YOLO_CLASSES
}

YOLO_CONFIG_PATH = LOCAL_ROOT / 'yolo_strategic_config.yaml'
with open(YOLO_CONFIG_PATH, 'w') as f:
  yaml.dump(yolo_yaml_content, f, default_flow_style=False)

print(f" Created YOLO config: {YOLO_CONFIG_PATH}")
!cat {YOLO_CONFIG_PATH}

In [ ]:
# 3.4 Dataset statistics and config

print(" YOLO statistics:")
if YOLO_DATA_YAML:
  yolo_counts = count_yolo_samples(LOCAL_YOLO)
  total_yolo = sum(yolo_counts.values())
  for split, count in yolo_counts.items():
    print(f"  {split}: {count:,} images")
  print(f"  TOTAL: {total_yolo:,} images")

  # Create config
  data_root = YOLO_DATA_YAML.parent
  new_yaml = {
    'path': str(data_root.absolute()),
    'train': 'images/train', 'val': 'images/val', 'test': 'images/test',
    'names': YOLO_CLASSES
  }
  YOLO_CONFIG_PATH = LOCAL_ROOT / 'yolo_strategic_config.yaml'
  with open(YOLO_CONFIG_PATH, 'w') as f:
    yaml.dump(new_yaml, f, default_flow_style=False)
  print(f" YOLO config: {YOLO_CONFIG_PATH}")
else:
  YOLO_CONFIG_PATH = None
  print(" Nie znaleziono struktury YOLO")

print("\n Statystyki Swin:")
if SWIN_TRAIN_DIR:
  swin_counts = count_swin_samples(LOCAL_SWIN)
  for split, classes in swin_counts.items():
    print(f"  {split}: {sum(classes.values()):,} images, {len(classes)} classes")
else:
  print(" Nie znaleziono struktury Swin")

## 4 YOLO Training

YOLOv8l detection model (120 epochs).

In [ ]:
# 4.1 Check for resume checkpoint

YOLO_LAST_PT = YOLO_CHECKPOINT_DIR / 'weights' / 'last.pt'
YOLO_BEST_PT = YOLO_CHECKPOINT_DIR / 'weights' / 'best.pt'
YOLO_RESUME = YOLO_LAST_PT.exists()

if YOLO_RESUME:
  print(f" Found YOLO checkpoint: {YOLO_LAST_PT}")
  print("  Resuming training!")
else:
  print("No checkpoint found. Starting new YOLO training.")

In [ ]:
# 4.2 Train YOLO

if YOLO_CONFIG_PATH is None:
  print(" No YOLO configuration found. Skipping this section.")
else:
  print(" Starting YOLO training...")
  print(f"  Model: {YOLO_CONFIG['model']}, Epochs: {YOLO_CONFIG['epochs']}")
  print(f"  Image size: {YOLO_CONFIG['imgsz']}, Batch: {YOLO_CONFIG['batch']}")
  print(f"  Resume: {YOLO_RESUME}")

  yolo_tracker = EmissionsTracker(
    project_name="VSHORAD_YOLO_Strategic",
    output_dir=str(LOGS_DIR), log_level='warning'
  )

  try:
    yolo_tracker.start()
    model = YOLO(str(YOLO_LAST_PT)) if YOLO_RESUME else YOLO(YOLO_CONFIG['model'])

    yolo_results = model.train(
      data=str(YOLO_CONFIG_PATH),
      epochs=YOLO_CONFIG['epochs'], patience=YOLO_CONFIG['patience'],
      batch=YOLO_CONFIG['batch'], imgsz=YOLO_CONFIG['imgsz'],
      device=YOLO_CONFIG['device'], optimizer=YOLO_CONFIG['optimizer'],
      lr0=YOLO_CONFIG['lr0'], lrf=YOLO_CONFIG['lrf'],
      momentum=YOLO_CONFIG['momentum'], weight_decay=YOLO_CONFIG['weight_decay'],
      warmup_epochs=YOLO_CONFIG['warmup_epochs'],
      box=YOLO_CONFIG['box'], cls=YOLO_CONFIG['cls'], dfl=YOLO_CONFIG['dfl'],
      mosaic=YOLO_CONFIG['mosaic'], mixup=YOLO_CONFIG['mixup'],
      copy_paste=YOLO_CONFIG['copy_paste'], degrees=YOLO_CONFIG['degrees'],
      translate=YOLO_CONFIG['translate'], scale=YOLO_CONFIG['scale'],
      shear=YOLO_CONFIG['shear'], flipud=YOLO_CONFIG['flipud'], fliplr=YOLO_CONFIG['fliplr'],
      hsv_h=YOLO_CONFIG['hsv_h'], hsv_s=YOLO_CONFIG['hsv_s'], hsv_v=YOLO_CONFIG['hsv_v'],
      erasing=YOLO_CONFIG['erasing'], workers=YOLO_CONFIG['workers'],
      project=YOLO_CONFIG['project'], name=YOLO_CONFIG['name'],
      exist_ok=True, pretrained=True, verbose=True, seed=42,
      save_period=YOLO_CONFIG['save_period'], resume=YOLO_RESUME,
    )

    yolo_emissions = yolo_tracker.stop()
    print("\n" + "" * 60)
    print(" YOLO TRAINING COMPLETE!")
    print(f"  Carbon: {yolo_emissions:.4f} kg CO2")

  except Exception as e:
    yolo_tracker.stop()
    print(f"\n ERROR: {e}")
    if 'model' in dir():
      emergency = YOLO_CHECKPOINT_DIR / 'weights' / 'emergency.pt'
      emergency.parent.mkdir(parents=True, exist_ok=True)
      model.save(str(emergency))
      print(f"  Emergency: {emergency}")
    raise

## 5 Swin Training

Swin-Base classification model (100 epochs).

In [ ]:
# 5.1 Swin augmentations

def get_swin_train_transform(img_size=384):
  return A.Compose([
    A.LongestMaxSize(max_size=int(img_size * 1.2)),
    A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.RandomCrop(height=img_size, width=img_size),
    A.OneOf([
      A.ToGray(p=1.0),
      A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=(-50, 30), val_shift_limit=(-40, 40), p=1.0),
    ], p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4, p=0.5),
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),
    A.OneOf([A.GaussianBlur(blur_limit=(3, 7)), A.MotionBlur(blur_limit=(3, 7))], p=0.2),
    A.OneOf([A.GaussNoise(var_limit=(10.0, 50.0)), A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5))], p=0.25),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, border_mode=cv2.BORDER_CONSTANT, value=0, p=0.5),
    A.CoarseDropout(max_holes=4, max_height=int(img_size*0.15), max_width=int(img_size*0.15), fill_value=0, p=0.15),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
  ])

def get_swin_val_transform(img_size=384):
  return A.Compose([
    A.LongestMaxSize(max_size=img_size),
    A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
  ])

class SwinAugmentedDataset(Dataset):
  def __init__(self, image_folder, transform):
    self.image_folder = image_folder
    self.transform = transform
  def __len__(self):
    return len(self.image_folder)
  def __getitem__(self, idx):
    img_path, label = self.image_folder.samples[idx]
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    augmented = self.transform(image=image)
    return augmented['image'], label

print(" Swin augmentations defined.")

In [ ]:
# 5.2 Prepare DataLoader

if SWIN_TRAIN_DIR is None:
  print(" No Swin train folder found.")
else:
  train_folder = datasets.ImageFolder(str(SWIN_TRAIN_DIR))
  val_folder = datasets.ImageFolder(str(SWIN_VAL_DIR)) if SWIN_VAL_DIR and SWIN_VAL_DIR.exists() else None

  SWIN_CLASSES = train_folder.classes
  SWIN_CLASS_TO_IDX = train_folder.class_to_idx
  SWIN_CONFIG['num_classes'] = len(SWIN_CLASSES)

  print(f" Swin: {len(SWIN_CLASSES)} classes, {len(train_folder)} train samples")

  train_transform = get_swin_train_transform(SWIN_CONFIG['img_size'])
  val_transform = get_swin_val_transform(SWIN_CONFIG['img_size'])

  train_dataset = SwinAugmentedDataset(train_folder, train_transform)
  val_dataset = SwinAugmentedDataset(val_folder, val_transform) if val_folder else None

  train_loader = DataLoader(train_dataset, batch_size=SWIN_CONFIG['batch_size'], shuffle=True,
               num_workers=SWIN_CONFIG['num_workers'], pin_memory=True, drop_last=True)
  val_loader = DataLoader(val_dataset, batch_size=SWIN_CONFIG['batch_size'], shuffle=False,
              num_workers=SWIN_CONFIG['num_workers'], pin_memory=True) if val_dataset else None

  print(f" DataLoadery: {len(train_loader)} train batches")

In [ ]:
# 5.3 Helper functions SWIN

def update_ema(model, ema_model, decay=0.9999):
  with torch.no_grad():
    for ema_param, param in zip(ema_model.parameters(), model.parameters()):
      ema_param.data.mul_(decay).add_(param.data, alpha=1 - decay)

def save_swin_checkpoint(model, ema_model, optimizer, scheduler, scaler, epoch, best_acc, path):
  checkpoint = {
    'epoch': epoch, 'model_state_dict': model.state_dict(),
    'ema_state_dict': ema_model.state_dict() if ema_model else None,
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
    'scaler_state_dict': scaler.state_dict() if scaler else None,
    'best_acc': best_acc, 'config': SWIN_CONFIG, 'classes': SWIN_CLASSES,
  }
  torch.save(checkpoint, path)

def load_swin_checkpoint(path, model, ema_model=None, optimizer=None, scheduler=None, scaler=None):
  checkpoint = torch.load(path, map_location=DEVICE)
  model.load_state_dict(checkpoint['model_state_dict'])
  if ema_model and checkpoint.get('ema_state_dict'): ema_model.load_state_dict(checkpoint['ema_state_dict'])
  if optimizer and checkpoint.get('optimizer_state_dict'): optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
  if scheduler and checkpoint.get('scheduler_state_dict'): scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
  if scaler and checkpoint.get('scaler_state_dict'): scaler.load_state_dict(checkpoint['scaler_state_dict'])
  return checkpoint['epoch'], checkpoint.get('best_acc', 0.0)

@torch.no_grad()
def evaluate_swin(model, loader, criterion, device):
  model.eval()
  total_loss, all_preds, all_labels = 0.0, [], []
  for images, labels in tqdm(loader, desc="Evaluating", leave=False):
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    loss = criterion(outputs, labels)
    total_loss += loss.item() * images.size(0)
    _, preds = outputs.max(1)
    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())
  avg_loss = total_loss / len(loader.dataset)
  accuracy = accuracy_score(all_labels, all_preds)
  return avg_loss, accuracy, np.array(all_preds), np.array(all_labels)

print(" Funkcje pomocnicze Swin zdefiniowane.")

In [ ]:
# 5.4 Swin checkpoint (resume)

SWIN_LAST_PTH = SWIN_CHECKPOINT_DIR / 'last.pth'
SWIN_BEST_PTH = SWIN_CHECKPOINT_DIR / 'best.pth'
SWIN_RESUME = SWIN_LAST_PTH.exists()

if SWIN_RESUME:
  print(f" Found Swin checkpoint: {SWIN_LAST_PTH}")
else:
  print("No checkpoint found. Starting new Swin training.")

In [ ]:
# 5.5 Train Swin

if SWIN_TRAIN_DIR is None:
  print(" No Swin data found.")
else:
  print(f" Training Swin: {SWIN_CONFIG['model_name']}, {SWIN_CONFIG['epochs']} epochs")

  swin_model = timm.create_model(SWIN_CONFIG['model_name'], pretrained=SWIN_CONFIG['pretrained'],
                  num_classes=SWIN_CONFIG['num_classes'], drop_path_rate=SWIN_CONFIG['drop_path_rate']).to(DEVICE)

  ema_model = timm.create_model(SWIN_CONFIG['model_name'], pretrained=False, num_classes=SWIN_CONFIG['num_classes']).to(DEVICE) if SWIN_CONFIG['use_ema'] else None
  if ema_model: ema_model.load_state_dict(swin_model.state_dict()); ema_model.eval()

  optimizer = torch.optim.AdamW(swin_model.parameters(), lr=SWIN_CONFIG['lr'], weight_decay=SWIN_CONFIG['weight_decay'])

  total_steps = len(train_loader) * SWIN_CONFIG['epochs']
  warmup_steps = len(train_loader) * SWIN_CONFIG['warmup_epochs']
  def lr_lambda(step):
    if step < warmup_steps: return step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return SWIN_CONFIG['min_lr'] / SWIN_CONFIG['lr'] + (1 - SWIN_CONFIG['min_lr'] / SWIN_CONFIG['lr']) * (1 + np.cos(np.pi * progress)) / 2
  scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

  criterion = nn.CrossEntropyLoss(label_smoothing=SWIN_CONFIG['label_smoothing'])
  mixup_fn = Mixup(mixup_alpha=SWIN_CONFIG['mixup_alpha'], cutmix_alpha=SWIN_CONFIG['cutmix_alpha'],
           prob=SWIN_CONFIG['mixup_prob'], switch_prob=0.5, mode='batch',
           label_smoothing=SWIN_CONFIG['label_smoothing'], num_classes=SWIN_CONFIG['num_classes'])
  scaler = GradScaler()

  start_epoch, best_acc = 0, 0.0
  if SWIN_RESUME:
    start_epoch, best_acc = load_swin_checkpoint(SWIN_LAST_PTH, swin_model, ema_model, optimizer, scheduler, scaler)
    start_epoch += 1
    print(f"  Resumed from epoch {start_epoch}, best_acc: {best_acc:.4f}")

  history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'lr': []}
  swin_tracker = EmissionsTracker(project_name="VSHORAD_Swin_Strategic", output_dir=str(LOGS_DIR), log_level='warning')

  try:
    swin_tracker.start()
    for epoch in range(start_epoch, SWIN_CONFIG['epochs']):
      swin_model.train()
      epoch_loss = 0.0
      pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{SWIN_CONFIG['epochs']}")
      for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        images, labels_mixed = mixup_fn(images, labels)
        optimizer.zero_grad()
        with autocast(): outputs = swin_model(images); loss = criterion(outputs, labels_mixed)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
        if ema_model: update_ema(swin_model, ema_model, SWIN_CONFIG['ema_decay'])
        epoch_loss += loss.item() * images.size(0)
        pbar.set_postfix({'loss': f"{loss.item():.4f}", 'lr': f"{scheduler.get_last_lr()[0]:.2e}"})

      avg_train_loss = epoch_loss / len(train_loader.dataset)
      history['train_loss'].append(avg_train_loss)
      history['lr'].append(scheduler.get_last_lr()[0])

      if val_loader:
        eval_model = ema_model if ema_model else swin_model
        val_loss, val_acc, _, _ = evaluate_swin(eval_model, val_loader, criterion, DEVICE)
        history['val_loss'].append(val_loss); history['val_acc'].append(val_acc)
        print(f"  Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")
        if val_acc > best_acc:
          best_acc = val_acc
          save_swin_checkpoint(swin_model, ema_model, optimizer, scheduler, scaler, epoch, best_acc, SWIN_BEST_PTH)
          print(f"  New best: {best_acc:.4f}")

      if (epoch + 1) % SWIN_CONFIG['save_period'] == 0:
        save_swin_checkpoint(swin_model, ema_model, optimizer, scheduler, scaler, epoch, best_acc, SWIN_CHECKPOINT_DIR / f'epoch_{epoch+1}.pth')
      save_swin_checkpoint(swin_model, ema_model, optimizer, scheduler, scaler, epoch, best_acc, SWIN_LAST_PTH)

    swin_emissions = swin_tracker.stop()
    print("\n" + "" * 60)
    print(f" SWIN COMPLETE! Best: {best_acc:.4f}, Carbon: {swin_emissions:.4f} kg CO2")

    with open(REPORTS_DIR / 'swin_training_history.json', 'w') as f:
      json.dump(history, f, indent=2)
  except Exception as e:
    swin_tracker.stop()
    print(f"\n ERROR: {e}")
    save_swin_checkpoint(swin_model, ema_model, optimizer, scheduler, scaler, epoch if 'epoch' in dir() else 0, best_acc, SWIN_CHECKPOINT_DIR / 'emergency.pth')
    raise

## 6 Quick Evaluation

Post-training verification of both models.

In [ ]:
# 6.1 YOLO EVALUATION

from ultralytics import YOLO

YOLO_BEST_PT = YOLO_CHECKPOINT_DIR / 'weights' / 'best.pt'
YOLO_LAST_PT = YOLO_CHECKPOINT_DIR / 'weights' / 'last.pt'

yolo_weights = YOLO_BEST_PT if YOLO_BEST_PT.exists() else YOLO_LAST_PT
if yolo_weights.exists():
  print(f" Evaluating YOLO: {yolo_weights.name}")
  yolo_eval = YOLO(str(yolo_weights))
  yolo_metrics = yolo_eval.val(
    data=str(YOLO_CONFIG_PATH),
    imgsz=YOLO_CONFIG['imgsz'],
    batch=YOLO_CONFIG['batch'] // 2,
    device=0,
  )
  print(f"\n YOLO Results:")
  print(f"  mAP@0.5:   {yolo_metrics.box.map50:.4f}")
  print(f"  mAP@0.5:0.95: {yolo_metrics.box.map:.4f}")
  print(f"  Precision:   {yolo_metrics.box.mp:.4f}")
  print(f"  Recall:    {yolo_metrics.box.mr:.4f}")
else:
  print(" No YOLO weights found.")

In [ ]:
# 6.2 SWIN EVALUATION

SWIN_BEST_PTH = SWIN_CHECKPOINT_DIR / 'best.pth'
SWIN_LAST_PTH = SWIN_CHECKPOINT_DIR / 'last.pth'

swin_weights = SWIN_BEST_PTH if SWIN_BEST_PTH.exists() else SWIN_LAST_PTH
if swin_weights.exists() and val_loader:
  # Load best model
  checkpoint = torch.load(swin_weights, map_location=DEVICE)
  swin_eval = timm.create_model(
    SWIN_CONFIG['model_name'], pretrained=False,
    num_classes=SWIN_CONFIG['num_classes'],
  ).to(DEVICE)

  # Handle both checkpoint formats
  state_dict = (checkpoint.get('ema_state_dict') or checkpoint.get('ema')
         or checkpoint.get('model_state_dict') or checkpoint.get('model'))
  swin_eval.load_state_dict(state_dict)
  swin_eval.eval()

  # Top-1 accuracy
  criterion = nn.CrossEntropyLoss()
  val_loss, val_acc, _, _ = evaluate_swin(swin_eval, val_loader, criterion, DEVICE)

  # Top-5 accuracy
  top5_correct = 0
  with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Top-5 eval", leave=False):
      images, labels = images.to(DEVICE), labels.to(DEVICE)
      _, top5_preds = swin_eval(images).topk(5, dim=1)
      for i, label in enumerate(labels):
        if label in top5_preds[i]:
          top5_correct += 1
  top5_acc = top5_correct / len(val_loader.dataset)

  print(f"\n Swin Results:")
  print(f"  Top-1 Accuracy: {val_acc:.4f}")
  print(f"  Top-5 Accuracy: {top5_acc:.4f}")
  print(f"  Val Loss:    {val_loss:.4f}")
else:
  print(" No Swin weights found or no validation data.")

In [ ]:
# 6.3 SUMMARY

print("\n" + "=" * 70)
print(" TRAINING COMPLETE — SUMMARY")
print("=" * 70)

print(f"\n All outputs saved to: {OUTPUT_ROOT}")
print(f"\n Checkpoints:")
!ls -lh {YOLO_CHECKPOINT_DIR}/weights/ 2>/dev/null || echo "  YOLO: not found"
!ls -lh {SWIN_CHECKPOINT_DIR}/*.pth 2>/dev/null || echo "  Swin: not found"

print(f"\n Pipeline finished successfully!")